# Stock Score Analyzer (web app)

**How to use:** click **Runtime → Run all**. After about a minute the last cell prints a public link. Open it to use the website.

- Edit `ideal_ranges.csv` (Files panel, left) and run the last cell again to apply your ranges.
- Edit formulas in cell 2 (`engine.py`, look for `# FORMULA`) and run **cell 2 and the last cell** again.
- The link stays alive only while the last cell is running (Colab stops idle sessions after a while).


In [ ]:
# 1. Install libraries and the free Cloudflare tunnel (gives your app a public link)
!pip install -q flask yfinance openpyxl pandas numpy
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared
!mkdir -p /content/stockscore/static
%cd /content/stockscore
print("Ready. Now run the next three cells (engine.py, app.py, index.html), then the launch cell.")


In [ ]:
%%writefile engine.py
"""
engine.py  -  the analysis engine (27 indicators, sector/market ranges, scoring, Excel export).

WHERE TO EDIT
  1. INDICATORS list   -> replace each `fn` with YOUR formula (marked  # FORMULA )
  2. ideal_ranges.csv  -> your ideal range per market / sector / indicator (auto-created on first run)
"""
import os, math, tempfile, datetime as dt
from dataclasses import dataclass
from typing import Callable, Optional

import numpy as np
import pandas as pd
import yfinance as yf

RANGES_CSV = "ideal_ranges.csv"

# ----------------------------------------------------------------------------
# 1. Timeframes  (price-based indicators use these; fundamentals use latest filings)
# ----------------------------------------------------------------------------
TIMEFRAMES = {
    # label: (history kwargs, bars per year used for annualising)
    "1 Day":   (dict(period="1d", interval="5m"), 78 * 252),
    "5 Days":  (dict(period="5d", interval="30m"), 13 * 252),
    "6 Months": (dict(period="6mo", interval="1d"), 252),
    "1 Year":  (dict(period="1y", interval="1d"), 252),
    "3 Years": (dict(period="3y", interval="1d"), 252),
    "4 Years": (dict(start=(dt.date.today() - dt.timedelta(days=365 * 4 + 1)).isoformat(), interval="1d"), 252),
}

RISK_FREE = {"India": 0.068, "Global": 0.043}   # used for Sharpe ratio; edit freely


# ----------------------------------------------------------------------------
# 2. Helpers
# ----------------------------------------------------------------------------
def num(x) -> Optional[float]:
    try:
        v = float(x)
        return v if math.isfinite(v) else None
    except (TypeError, ValueError):
        return None


def ratio(a, b, scale=1.0) -> Optional[float]:
    a, b = num(a), num(b)
    if a is None or b is None or b == 0:
        return None
    return a / b * scale


def pct(x) -> Optional[float]:
    """yfinance gives fractions (0.25) -> we show percent (25)."""
    v = num(x)
    return None if v is None else v * 100


@dataclass
class Ctx:
    """Everything an indicator formula can use."""
    info: dict
    income: pd.DataFrame          # annual income statement (rows = items, cols = dates)
    balance: pd.DataFrame         # annual balance sheet
    close: pd.Series              # closing prices for the selected timeframe
    bars_per_year: int
    market: str                   # "India" or "Global"


def stmt_row(df: pd.DataFrame, *names) -> Optional[float]:
    if df is None or df.empty:
        return None
    for n in names:
        if n in df.index:
            s = df.loc[n].dropna()
            if len(s):
                return num(s.iloc[0])
    return None


def returns(c: Ctx) -> pd.Series:
    return c.close.pct_change().dropna()


# ----------------------------------------------------------------------------
# 3. THE 27 INDICATORS  -  replace formulas with yours.  Each fn(ctx) -> float | None
#    Units: ratios as plain numbers, anything "%" is expressed as a percent (25 = 25%).
# ----------------------------------------------------------------------------
@dataclass
class Indicator:
    key: str
    name: str
    group: str
    unit: str
    formula: str                      # human-readable text shown in the Excel file
    fn: Callable[[Ctx], Optional[float]]
    default_range: tuple              # (low, high); None = open ended


def _roic(c: Ctx):
    ebit = stmt_row(c.income, "EBIT", "Operating Income")
    tax_rate = num(stmt_row(c.income, "Tax Rate For Calcs")) or 0.21
    invested = stmt_row(c.balance, "Invested Capital")
    if ebit is None or invested in (None, 0):
        return None
    return ebit * (1 - tax_rate) / invested * 100


def _interest_cov(c: Ctx):
    ebit = stmt_row(c.income, "EBIT", "Operating Income")
    interest = stmt_row(c.income, "Interest Expense")
    if ebit is None or not interest:
        return None
    return ebit / abs(interest)


def _fcf_yield(c: Ctx):
    return ratio(c.info.get("freeCashflow"), c.info.get("marketCap"), 100)


def _div_yield(c: Ctx):
    price = c.info.get("currentPrice") or c.info.get("regularMarketPrice")
    r = ratio(c.info.get("dividendRate"), price, 100)
    return 0.0 if r is None and c.info.get("dividendRate") is None else r


def _net_debt_ebitda(c: Ctx):
    debt, cash = num(c.info.get("totalDebt")), num(c.info.get("totalCash"))
    if debt is None or cash is None:
        return None
    return ratio(debt - cash, c.info.get("ebitda"))


def _total_return(c: Ctx):
    s = c.close.dropna()
    return None if len(s) < 2 else (s.iloc[-1] / s.iloc[0] - 1) * 100


def _volatility(c: Ctx):
    r = returns(c)
    return None if len(r) < 2 else r.std() * math.sqrt(c.bars_per_year) * 100


def _max_drawdown(c: Ctx):
    s = c.close.dropna()
    return None if len(s) < 2 else ((s / s.cummax()) - 1).min() * 100


def _rsi(c: Ctx, n=14):
    d = c.close.diff().dropna()
    if len(d) < n + 1:
        return None
    up, dn = d.clip(lower=0), (-d).clip(lower=0)
    rs = up.ewm(alpha=1 / n, adjust=False).mean().iloc[-1] / max(dn.ewm(alpha=1 / n, adjust=False).mean().iloc[-1], 1e-12)
    return 100 - 100 / (1 + rs)


def _sharpe(c: Ctx):
    r = returns(c)
    if len(r) < 5 or r.std() == 0:
        return None
    rf = RISK_FREE.get(c.market, 0.04) / c.bars_per_year
    return (r.mean() - rf) / r.std() * math.sqrt(c.bars_per_year)


INDICATORS = [
    # ---- Valuation -------------------------------------------------------
    Indicator("pe", "P/E (trailing)", "Valuation", "x", "Price / EPS (TTM)",
              lambda c: num(c.info.get("trailingPE")), (10, 25)),                       # FORMULA
    Indicator("fwd_pe", "Forward P/E", "Valuation", "x", "Price / forward EPS",
              lambda c: num(c.info.get("forwardPE")), (8, 22)),                         # FORMULA
    Indicator("pb", "Price / Book", "Valuation", "x", "Price / book value per share",
              lambda c: num(c.info.get("priceToBook")), (1, 4)),                        # FORMULA
    Indicator("ps", "Price / Sales", "Valuation", "x", "Market cap / revenue (TTM)",
              lambda c: num(c.info.get("priceToSalesTrailing12Months")), (1, 5)),       # FORMULA
    Indicator("ev_ebitda", "EV / EBITDA", "Valuation", "x", "Enterprise value / EBITDA",
              lambda c: num(c.info.get("enterpriseToEbitda")), (6, 15)),                # FORMULA
    Indicator("peg", "PEG ratio", "Valuation", "x", "P/E divided by earnings growth",
              lambda c: num(c.info.get("trailingPegRatio") or c.info.get("pegRatio")), (0.5, 1.5)),  # FORMULA
    # ---- Profitability ---------------------------------------------------
    Indicator("roe", "Return on equity", "Profitability", "%", "Net income / shareholder equity",
              lambda c: pct(c.info.get("returnOnEquity")), (15, None)),                 # FORMULA
    Indicator("roa", "Return on assets", "Profitability", "%", "Net income / total assets",
              lambda c: pct(c.info.get("returnOnAssets")), (5, None)),                  # FORMULA
    Indicator("roic", "Return on invested capital", "Profitability", "%", "EBIT x (1 - tax) / invested capital",
              _roic, (10, None)),                                                       # FORMULA
    Indicator("gross_m", "Gross margin", "Profitability", "%", "Gross profit / revenue",
              lambda c: pct(c.info.get("grossMargins")), (30, None)),                   # FORMULA
    Indicator("op_m", "Operating margin", "Profitability", "%", "Operating income / revenue",
              lambda c: pct(c.info.get("operatingMargins")), (15, None)),               # FORMULA
    Indicator("net_m", "Net margin", "Profitability", "%", "Net income / revenue",
              lambda c: pct(c.info.get("profitMargins")), (10, None)),                  # FORMULA
    # ---- Growth ----------------------------------------------------------
    Indicator("rev_g", "Revenue growth (YoY)", "Growth", "%", "Latest quarter revenue vs same quarter last year",
              lambda c: pct(c.info.get("revenueGrowth")), (5, None)),                   # FORMULA
    Indicator("earn_g", "Earnings growth (YoY)", "Growth", "%", "Latest quarter earnings vs same quarter last year",
              lambda c: pct(c.info.get("earningsGrowth")), (5, None)),                  # FORMULA
    Indicator("eps_g", "Quarterly earnings growth", "Growth", "%", "Quarterly net income growth",
              lambda c: pct(c.info.get("earningsQuarterlyGrowth")), (5, None)),         # FORMULA
    # ---- Leverage and liquidity -----------------------------------------
    Indicator("de", "Debt / Equity", "Financial health", "x", "Total debt / shareholder equity",
              lambda c: None if num(c.info.get("debtToEquity")) is None else num(c.info.get("debtToEquity")) / 100,
              (0, 1)),                                                                  # FORMULA
    Indicator("current", "Current ratio", "Financial health", "x", "Current assets / current liabilities",
              lambda c: num(c.info.get("currentRatio")), (1.2, 3)),                     # FORMULA
    Indicator("quick", "Quick ratio", "Financial health", "x", "(Cash + receivables) / current liabilities",
              lambda c: num(c.info.get("quickRatio")), (1, 2.5)),                       # FORMULA
    Indicator("int_cov", "Interest coverage", "Financial health", "x", "EBIT / interest expense",
              _interest_cov, (3, None)),                                                # FORMULA
    Indicator("nd_ebitda", "Net debt / EBITDA", "Financial health", "x", "(Total debt - cash) / EBITDA",
              _net_debt_ebitda, (0, 3)),                                                # FORMULA
    # ---- Cash returns ----------------------------------------------------
    Indicator("fcf_y", "Free cash flow yield", "Cash returns", "%", "Free cash flow / market cap",
              _fcf_yield, (3, None)),                                                   # FORMULA
    Indicator("div_y", "Dividend yield", "Cash returns", "%", "Annual dividend / price",
              _div_yield, (1, None)),                                                   # FORMULA
    # ---- Price behaviour (uses the selected timeframe) ------------------
    Indicator("ret", "Price return", "Price behaviour", "%", "Last close / first close - 1",
              _total_return, (0, None)),                                                # FORMULA
    Indicator("vol", "Volatility (annualised)", "Price behaviour", "%", "Std dev of returns x sqrt(bars/year)",
              _volatility, (0, 35)),                                                    # FORMULA
    Indicator("mdd", "Maximum drawdown", "Price behaviour", "%", "Worst peak-to-trough fall",
              _max_drawdown, (-25, 0)),                                                 # FORMULA
    Indicator("rsi", "RSI (14)", "Price behaviour", "", "Wilder RSI, 14 bars",
              _rsi, (40, 65)),                                                          # FORMULA
    Indicator("sharpe", "Sharpe ratio", "Price behaviour", "", "(Mean return - risk free) / std dev",
              _sharpe, (1, None)),                                                      # FORMULA
]
assert len(INDICATORS) == 27, f"Expected 27 indicators, found {len(INDICATORS)}"


# ----------------------------------------------------------------------------
# 4. Ideal ranges: CSV  (market, sector, indicator_key, low, high)
#    Lookup order: exact market+sector -> market + "*" -> built-in default
# ----------------------------------------------------------------------------
SECTORS = ["Technology", "Healthcare", "Financial Services", "Consumer Cyclical", "Consumer Defensive",
           "Industrials", "Energy", "Utilities", "Real Estate", "Basic Materials", "Communication Services"]


def create_template_csv():
    rows = []
    for m in ("Global", "India"):
        for s in SECTORS:
            for ind in INDICATORS:
                lo, hi = ind.default_range
                rows.append(dict(market=m, sector=s, indicator_key=ind.key, indicator=ind.name,
                                 low="" if lo is None else lo, high="" if hi is None else hi))
    pd.DataFrame(rows).to_csv(RANGES_CSV, index=False)


def load_ranges() -> dict:
    if not os.path.exists(RANGES_CSV):
        create_template_csv()
    df = pd.read_csv(RANGES_CSV)
    out = {}
    for r in df.itertuples():
        out[(str(r.market).strip(), str(r.sector).strip(), str(r.indicator_key).strip())] = (num(r.low), num(r.high))
    return out


def get_range(ranges, market, sector, ind: Indicator):
    for k in ((market, sector, ind.key), (market, "*", ind.key)):
        if k in ranges:
            return ranges[k]
    return ind.default_range


# ----------------------------------------------------------------------------
# 5. Scoring:  inside range = 100; outside decays linearly with distance
# ----------------------------------------------------------------------------
def score_value(v, lo, hi):
    if v is None:
        return None, "No data"
    if lo is not None and v < lo:
        dist, status, ref = lo - v, "Below", lo
    elif hi is not None and v > hi:
        dist, status, ref = v - hi, "Above", hi
    else:
        return 100.0, "Within"
    width = (hi - lo) if (lo is not None and hi is not None) else abs(ref)
    scale = max(abs(width), abs(ref) * 0.5, 1e-9)
    return max(0.0, 100 - 100 * dist / scale), status


def rating(s):
    if s >= 80: return "Excellent"
    if s >= 65: return "Strong"
    if s >= 50: return "Average"
    if s >= 35: return "Weak"
    return "Poor"


# ----------------------------------------------------------------------------
# 6. Data + analysis
# ----------------------------------------------------------------------------
def resolve_ticker(text: str):
    t = text.strip().upper().replace(" ", "")
    if not t:
        raise ValueError("Enter a ticker, for example NVDA or RELIANCE.NS")
    candidates = [t] if "." in t else [t, t + ".NS", t + ".BO"]
    for cand in candidates:
        try:
            h = yf.Ticker(cand).history(period="5d")
            if not h.empty:
                return cand
        except Exception:
            pass
    try:                                           # fall back to name search ("nvidia")
        for q in yf.Search(text.strip(), max_results=5).quotes:
            if q.get("symbol") and q.get("quoteType") in ("EQUITY", "ETF"):
                return q["symbol"]
    except Exception:
        pass
    raise ValueError(f"Could not find '{text}'. Try the exact Yahoo ticker (e.g. TCS.NS for India).")


def detect_market(symbol: str, info: dict) -> str:
    if symbol.endswith((".NS", ".BO")) or info.get("country") == "India":
        return "India"
    return "Global"


def analyze(user_text: str, timeframe: str):
    symbol = resolve_ticker(user_text)
    tk = yf.Ticker(symbol)
    info = tk.info or {}
    market = detect_market(symbol, info)
    sector = info.get("sector") or "Unknown"

    kwargs, bpy = TIMEFRAMES[timeframe]
    hist = tk.history(auto_adjust=True, **kwargs)
    if hist.empty:
        raise ValueError("No price history returned for this timeframe.")
    try:
        income, balance = tk.income_stmt, tk.balance_sheet
    except Exception:
        income, balance = pd.DataFrame(), pd.DataFrame()

    ctx = Ctx(info, income, balance, hist["Close"], bpy, market)
    ranges = load_ranges()

    rows = []
    for ind in INDICATORS:
        try:
            v = ind.fn(ctx)
        except Exception:
            v = None
        lo, hi = get_range(ranges, market, sector, ind)
        s, status = score_value(v, lo, hi)
        rows.append(dict(Group=ind.group, Indicator=ind.name, Value=v, Unit=ind.unit, Low=lo, High=hi,
                         Status=status, Score=s, Formula=ind.formula))
    df = pd.DataFrame(rows)

    valid = df["Score"].dropna()
    overall = float(valid.mean()) if len(valid) else 0.0
    meta = dict(symbol=symbol, name=info.get("longName") or info.get("shortName") or symbol, sector=sector,
                industry=info.get("industry", "-"), market=market, currency=info.get("currency", ""),
                timeframe=timeframe, price=num(hist["Close"].iloc[-1]), overall=overall, rating=rating(overall),
                coverage=f"{len(valid)} of {len(INDICATORS)}",
                generated=dt.datetime.now().strftime("%d %b %Y, %H:%M"))
    return meta, df


def fmt_range(lo, hi, unit):
    u = unit if unit in ("%", "x") else ""
    if lo is None and hi is None: return "Any"
    if lo is None: return f"up to {hi:g}{u}"
    if hi is None: return f"{lo:g}{u} and above"
    return f"{lo:g}{u} to {hi:g}{u}"


def fmt_val(v, unit):
    if v is None: return "N/A"
    u = unit if unit in ("%", "x") else ""
    return f"{v:,.2f}{u}"


# ----------------------------------------------------------------------------
# 7. Excel export
# ----------------------------------------------------------------------------
def to_excel(meta: dict, df: pd.DataFrame) -> str:
    from openpyxl import Workbook
    from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
    from openpyxl.utils import get_column_letter

    wb = Workbook()
    ink, paper = "14213D", "F4F6F9"
    head_fill = PatternFill("solid", fgColor=ink)
    fills = {"Within": "D7F0E3", "Below": "FBE3C8", "Above": "FBE3C8", "No data": "E8EAED"}
    thin = Side(style="thin", color="D5D9E0")

    ws = wb.active
    ws.title = "Summary"
    ws["A1"] = f"{meta['name']} ({meta['symbol']})"
    ws["A1"].font = Font(size=18, bold=True, color=ink)
    items = [("Overall score", f"{meta['overall']:.1f} / 100"), ("Rating", meta["rating"]),
             ("Sector", meta["sector"]), ("Industry", meta["industry"]), ("Market", meta["market"]),
             ("Timeframe", meta["timeframe"]), ("Last price", f"{meta['price']:,.2f} {meta['currency']}"),
             ("Indicators scored", meta["coverage"]), ("Generated", meta["generated"])]
    for i, (k, v) in enumerate(items, start=3):
        ws.cell(i, 1, k).font = Font(bold=True)
        ws.cell(i, 2, v)
    r0 = len(items) + 5
    ws.cell(r0, 1, "Score by group").font = Font(bold=True, size=12, color=ink)
    g = df.groupby("Group", sort=False)["Score"].mean()
    for i, (name, val) in enumerate(g.items(), start=r0 + 1):
        ws.cell(i, 1, name)
        ws.cell(i, 2, None if pd.isna(val) else round(float(val), 1))
    ws.column_dimensions["A"].width = 26
    ws.column_dimensions["B"].width = 28

    ws2 = wb.create_sheet("Indicators")
    heads = ["#", "Group", "Indicator", "Value", "Ideal range", "Status", "Score (0-100)", "Formula"]
    for c, h in enumerate(heads, 1):
        cell = ws2.cell(1, c, h)
        cell.font, cell.fill = Font(bold=True, color="FFFFFF"), head_fill
        cell.alignment = Alignment(horizontal="center", vertical="center")
    for i, r in enumerate(df.itertuples(), start=1):
        vals = [i, r.Group, r.Indicator, fmt_val(r.Value, r.Unit), fmt_range(r.Low, r.High, r.Unit),
                r.Status, None if pd.isna(r.Score) else round(r.Score, 1), r.Formula]
        for c, v in enumerate(vals, 1):
            cell = ws2.cell(i + 1, c, v)
            cell.border = Border(top=thin, bottom=thin, left=thin, right=thin)
            if c == 6:
                cell.fill = PatternFill("solid", fgColor=fills.get(r.Status, paper))
            if c in (1, 4, 5, 6, 7):
                cell.alignment = Alignment(horizontal="center")
    for c, w in enumerate([5, 18, 30, 14, 22, 12, 14, 48], 1):
        ws2.column_dimensions[get_column_letter(c)].width = w
    ws2.freeze_panes = "A2"

    path = os.path.join(tempfile.gettempdir(), f"{meta['symbol']}_{meta['timeframe'].replace(' ', '')}_score.xlsx")
    wb.save(path)
    return path




In [ ]:
%%writefile app.py
"""
app.py  -  web server.   Run locally:  python app.py   ->  open http://localhost:5000
Production (Render / Railway / any server):  gunicorn app:app
"""
import io, os, time, math
from flask import Flask, jsonify, request, send_file, send_from_directory

import engine

app = Flask(__name__, static_folder="static", static_url_path="/static")

_CACHE, TTL = {}, 600          # keep results 10 minutes so repeat clicks and Excel export don't re-fetch


def clean(v):
    """NaN / numpy values -> plain JSON-safe values."""
    if v is None:
        return None
    try:
        f = float(v)
        return None if math.isnan(f) or math.isinf(f) else f
    except (TypeError, ValueError):
        return v


def run_analysis(ticker: str, timeframe: str):
    key = (ticker.strip().upper(), timeframe)
    hit = _CACHE.get(key)
    if hit and time.time() - hit[0] < TTL:
        return hit[1], hit[2]
    meta, df = engine.analyze(ticker, timeframe)
    _CACHE[key] = (time.time(), meta, df)
    return meta, df


def to_payload(meta, df):
    rows = []
    for i, r in enumerate(df.itertuples(), 1):
        rows.append(dict(
            n=i, group=r.Group, name=r.Indicator, value=clean(r.Value), unit=r.Unit,
            valueText=engine.fmt_val(clean(r.Value), r.Unit),
            low=clean(r.Low), high=clean(r.High), rangeText=engine.fmt_range(clean(r.Low), clean(r.High), r.Unit),
            status=r.Status, score=clean(r.Score), formula=r.Formula))
    groups = [dict(name=n, score=clean(v)) for n, v in df.groupby("Group", sort=False)["Score"].mean().items()]
    return dict(meta={k: clean(v) for k, v in meta.items()}, indicators=rows, groups=groups)


@app.get("/")
def home():
    return send_from_directory("static", "index.html")


@app.get("/api/analyze")
def api_analyze():
    ticker = request.args.get("ticker", "")
    tf = request.args.get("timeframe", "1 Year")
    if tf not in engine.TIMEFRAMES:
        return jsonify(error=f"Unknown timeframe '{tf}'."), 400
    try:
        meta, df = run_analysis(ticker, tf)
        return jsonify(to_payload(meta, df))
    except ValueError as e:
        return jsonify(error=str(e)), 404
    except Exception as e:                       # network problems, Yahoo rate limits, etc.
        return jsonify(error=f"Data provider problem: {e}. Please try again in a moment."), 502


@app.get("/api/search")
def api_search():
    q = request.args.get("q", "").strip()
    if len(q) < 2:
        return jsonify([])
    try:
        import yfinance as yf
        out = []
        for h in yf.Search(q, max_results=7).quotes:
            if h.get("quoteType") in ("EQUITY", "ETF") and h.get("symbol"):
                out.append(dict(symbol=h["symbol"], name=h.get("longname") or h.get("shortname") or "",
                                exchange=h.get("exchDisp") or h.get("exchange") or ""))
        return jsonify(out)
    except Exception:
        return jsonify([])


@app.get("/api/export")
def api_export():
    ticker = request.args.get("ticker", "")
    tf = request.args.get("timeframe", "1 Year")
    try:
        meta, df = run_analysis(ticker, tf)
        path = engine.to_excel(meta, df)
        return send_file(path, as_attachment=True, download_name=os.path.basename(path))
    except Exception as e:
        return jsonify(error=str(e)), 400


@app.get("/api/meta")
def api_meta():
    return jsonify(timeframes=list(engine.TIMEFRAMES), indicators=len(engine.INDICATORS))


if __name__ == "__main__":
    engine.load_ranges()                                    # creates ideal_ranges.csv on first run
    app.run(host="0.0.0.0", port=int(os.environ.get("PORT", 5000)), debug=False)


In [ ]:
%%writefile static/index.html
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Stock Score Analyzer</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=Bricolage+Grotesque:opsz,wght@12..96,500;12..96,700&family=IBM+Plex+Sans:wght@400;500;600&display=swap" rel="stylesheet">
<style>
:root{
  --ink:#14213d; --paper:#f3f5f8; --card:#fff; --line:#d9dee7; --muted:#5c6679;
  --good:#1c8a5a; --mid:#e0912f; --bad:#c2452f; --none:#c4cad6; --accent:#2b59c3;
  --display:'Bricolage Grotesque',Georgia,sans-serif; --body:'IBM Plex Sans',system-ui,sans-serif;
}
*{box-sizing:border-box}
html{scroll-behavior:smooth}
body{margin:0;background:var(--paper);color:var(--ink);font-family:var(--body);line-height:1.5}
.wrap{max-width:1160px;margin:0 auto;padding:0 22px}

/* ---------- top band ---------- */
header{background:var(--ink);color:#fff;padding:34px 0 38px}
header h1{font-family:var(--display);font-size:clamp(2rem,4.5vw,3.1rem);margin:0;letter-spacing:-.02em;line-height:1.05}
header p{margin:10px 0 0;color:#b9c4dc;max-width:620px}
.search{margin-top:26px;display:grid;grid-template-columns:minmax(220px,1.4fr) auto auto;gap:12px;align-items:stretch}
.field{position:relative}
.field input{width:100%;height:50px;border:0;border-radius:10px;padding:0 16px;font:500 1.05rem var(--body);color:var(--ink);background:#fff}
.field input:focus,.seg button:focus-visible,.go:focus-visible,.chip-btn:focus-visible{outline:3px solid #8fb0ff;outline-offset:2px}
.suggest{position:absolute;top:56px;left:0;right:0;background:#fff;border-radius:10px;box-shadow:0 14px 34px rgba(8,16,40,.28);z-index:10;overflow:hidden;display:none}
.suggest button{display:flex;justify-content:space-between;gap:12px;width:100%;border:0;background:#fff;padding:11px 16px;text-align:left;font:inherit;color:var(--ink);cursor:pointer;border-bottom:1px solid var(--line)}
.suggest button:last-child{border-bottom:0}
.suggest button:hover,.suggest button.on{background:#eef2fb}
.suggest small{color:var(--muted)}
.seg{display:flex;background:rgba(255,255,255,.12);border-radius:10px;padding:4px;gap:2px;flex-wrap:wrap}
.seg button{border:0;background:transparent;color:#d3dcf0;font:500 .92rem var(--body);padding:0 14px;min-height:42px;border-radius:7px;cursor:pointer}
.seg button[aria-pressed=true]{background:#fff;color:var(--ink);font-weight:600}
.go{border:0;height:50px;padding:0 28px;border-radius:10px;background:#5b8cff;color:#fff;font:600 1rem var(--body);cursor:pointer}
.go:hover{background:#7aa0ff}.go:disabled{opacity:.6;cursor:wait}
.recent{margin-top:14px;display:flex;gap:8px;flex-wrap:wrap;align-items:center;font-size:.85rem;color:#9fb0d4;min-height:30px}
.chip-btn{border:1px solid rgba(255,255,255,.28);background:transparent;color:#e7edfb;border-radius:999px;padding:3px 12px;font:inherit;cursor:pointer}
.chip-btn:hover{background:rgba(255,255,255,.12)}

/* ---------- results ---------- */
main{padding:6px 0 60px}
.panel{background:var(--card);border:1px solid var(--line);border-radius:14px;padding:24px 28px;margin-top:18px}
.hero{display:grid;grid-template-columns:auto 1fr;gap:34px;align-items:center}
.big{font-family:var(--display);font-size:5.6rem;line-height:.9;font-weight:700;letter-spacing:-.03em}
.big small{font-size:1.3rem;font-weight:500;color:var(--muted);letter-spacing:0}
.rate{font-weight:600;font-size:1.1rem;margin-top:8px}
.name{font-family:var(--display);font-size:1.7rem;font-weight:700;line-height:1.15}
.sym{color:var(--muted);font-weight:500;font-size:1.1rem;margin-left:6px}
.tags{display:flex;flex-wrap:wrap;gap:8px;margin-top:10px}
.tags span{border:1px solid var(--line);border-radius:999px;padding:3px 12px;font-size:.85rem;background:var(--paper)}
.strip{display:grid;grid-template-columns:repeat(27,1fr);gap:3px;margin-top:24px}
.strip i{height:28px;border-radius:3px;cursor:pointer}
.strip i:hover{transform:scaleY(1.25)}
.legend{display:flex;gap:18px;flex-wrap:wrap;margin-top:12px;font-size:.82rem;color:var(--muted)}
.legend b{display:inline-block;width:11px;height:11px;border-radius:3px;margin-right:6px;vertical-align:-1px}
.groups{display:grid;grid-template-columns:repeat(auto-fit,minmax(150px,1fr));gap:14px;margin-top:24px}
.groups div{border-left:3px solid var(--accent);padding-left:12px}
.groups span{display:block;font-size:.82rem;color:var(--muted)}
.groups strong{font-family:var(--display);font-size:1.6rem}
.tools{display:flex;justify-content:space-between;align-items:center;gap:12px;flex-wrap:wrap;margin-bottom:14px}
.filters{display:flex;gap:8px;flex-wrap:wrap}
.filters button{border:1px solid var(--line);background:#fff;border-radius:999px;padding:5px 14px;font:500 .88rem var(--body);cursor:pointer;color:var(--ink)}
.filters button[aria-pressed=true]{background:var(--ink);color:#fff;border-color:var(--ink)}
.xl{font:500 .88rem var(--body);color:var(--accent);text-decoration:none;border:1px solid var(--line);padding:6px 14px;border-radius:8px;background:#fff}
.xl:hover{border-color:var(--accent)}
.tablewrap{overflow-x:auto}
table{width:100%;border-collapse:collapse;font-size:.93rem;min-width:640px}
th{text-align:left;color:var(--muted);font-weight:500;padding:8px 10px;border-bottom:2px solid var(--ink);white-space:nowrap}
td{padding:10px;border-bottom:1px solid var(--line);vertical-align:middle}
tr.grp td{background:var(--paper);font-weight:600;color:var(--accent);padding:7px 10px}
tr.row{cursor:pointer}tr.row:hover td{background:#f8f9fc}
tr.row.flash td{background:#fff6d6}
tr.detail td{background:#f8f9fc;color:var(--muted);font-size:.88rem;padding:8px 10px 12px 44px}
.num{font-variant-numeric:tabular-nums}
.chip{display:inline-block;padding:2px 10px;border-radius:999px;font-size:.78rem;font-weight:600;white-space:nowrap}
.c-Within{background:#d7f0e3;color:#11603f}.c-Below,.c-Above{background:#fbe3c8;color:#8a4d0b}.c-No{background:#e8eaed;color:#4a5262}
.bar{display:inline-block;width:84px;height:6px;border-radius:3px;background:var(--line);margin-right:8px;vertical-align:middle;overflow:hidden}
.bar u{display:block;height:100%;text-decoration:none}
.empty{color:var(--muted);text-align:center;padding:46px 20px}
.empty b{display:block;color:var(--ink);font-family:var(--display);font-size:1.3rem;margin-bottom:4px}
.err{border-left:4px solid var(--bad);background:#fff3f0;color:#7a2616}
.load{display:flex;align-items:center;gap:14px;color:var(--muted)}
.spin{width:22px;height:22px;border:3px solid var(--line);border-top-color:var(--accent);border-radius:50%;animation:sp .8s linear infinite}
@keyframes sp{to{transform:rotate(360deg)}}
footer{color:var(--muted);font-size:.82rem;padding:0 0 40px}
@media(max-width:820px){.search{grid-template-columns:1fr}.hero{grid-template-columns:1fr;gap:16px}.big{font-size:4rem}.strip i{height:22px}}
@media(prefers-reduced-motion:reduce){.spin{animation-duration:2s}.strip i:hover{transform:none}}
</style>
</head>
<body>
<header>
  <div class="wrap">
    <h1>Stock Score Analyzer</h1>
    <p>Search a company. We find its sector and market, calculate 27 indicators, and score each one against the ideal range for that sector.</p>
    <form class="search" id="form" autocomplete="off">
      <div class="field">
        <input id="q" type="text" placeholder="Nvidia, AAPL, RELIANCE.NS, TCS.NS" aria-label="Ticker or company name" spellcheck="false">
        <div class="suggest" id="suggest" role="listbox"></div>
      </div>
      <div class="seg" id="tf" role="group" aria-label="Timeframe"></div>
      <button class="go" id="go" type="submit">Analyze</button>
    </form>
    <div class="recent" id="recent"></div>
  </div>
</header>

<main class="wrap" id="out">
  <div class="panel empty"><b>Search a ticker to begin</b>Indian stocks use .NS (NSE) or .BO (BSE), for example INFY.NS.</div>
</main>
<footer class="wrap">Data from Yahoo Finance and may be delayed or incomplete. Scores are a screening aid, not investment advice.</footer>

<script>
const TFS = ["1 Day","5 Days","6 Months","1 Year","3 Years","4 Years"];
let tf = "1 Year", current = null, filter = "all", timer = null;
const $ = id => document.getElementById(id);
const esc = s => String(s ?? "").replace(/[&<>"']/g, c => ({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#39;'}[c]));
const col = s => s == null ? "var(--none)" : s >= 70 ? "var(--good)" : s >= 40 ? "var(--mid)" : "var(--bad)";

/* ---------- timeframe buttons ---------- */
function drawTF(){
  $("tf").innerHTML = TFS.map(t => `<button type="button" aria-pressed="${t===tf}" data-t="${t}">${t}</button>`).join("");
}
$("tf").addEventListener("click", e => {
  const b = e.target.closest("button"); if(!b) return;
  tf = b.dataset.t; drawTF();
  if (current) analyze(current);          // re-run instantly when timeframe changes
});

/* ---------- recent searches ---------- */
function recents(){ try{ return JSON.parse(localStorage.getItem("recent")||"[]"); }catch{ return []; } }
function pushRecent(sym){
  try{ const r=[sym,...recents().filter(x=>x!==sym)].slice(0,6); localStorage.setItem("recent",JSON.stringify(r)); }catch{}
  drawRecent();
}
function drawRecent(){
  const r = recents();
  $("recent").innerHTML = r.length ? "Recent: " + r.map(s=>`<button type="button" class="chip-btn" data-s="${esc(s)}">${esc(s)}</button>`).join("") : "";
}
$("recent").addEventListener("click", e => { const b=e.target.closest("button"); if(b){ $("q").value=b.dataset.s; analyze(b.dataset.s); }});

/* ---------- suggestions ---------- */
$("q").addEventListener("input", () => {
  clearTimeout(timer);
  const v = $("q").value.trim();
  if (v.length < 2) { $("suggest").style.display = "none"; return; }
  timer = setTimeout(async () => {
    try{
      const r = await (await fetch("/api/search?q=" + encodeURIComponent(v))).json();
      $("suggest").innerHTML = r.map(x => `<button type="button" data-s="${esc(x.symbol)}"><span><b>${esc(x.symbol)}</b> ${esc(x.name)}</span><small>${esc(x.exchange)}</small></button>`).join("");
      $("suggest").style.display = r.length ? "block" : "none";
    }catch{ $("suggest").style.display = "none"; }
  }, 280);
});
$("suggest").addEventListener("click", e => { const b=e.target.closest("button"); if(b){ $("q").value=b.dataset.s; $("suggest").style.display="none"; analyze(b.dataset.s); }});
document.addEventListener("click", e => { if(!e.target.closest(".field")) $("suggest").style.display="none"; });
$("form").addEventListener("submit", e => { e.preventDefault(); $("suggest").style.display="none"; analyze($("q").value.trim()); });

/* ---------- analysis ---------- */
async function analyze(ticker){
  if (!ticker) return;
  current = ticker; filter = "all";
  $("go").disabled = true;
  $("out").innerHTML = `<div class="panel load"><div class="spin"></div><div>Fetching data for <b>${esc(ticker)}</b> and scoring 27 indicators. This takes a few seconds.</div></div>`;
  try{
    const res = await fetch(`/api/analyze?ticker=${encodeURIComponent(ticker)}&timeframe=${encodeURIComponent(tf)}`);
    const data = await res.json();
    if(!res.ok) throw new Error(data.error || "Something went wrong.");
    render(data);
    pushRecent(data.meta.symbol);
    history.replaceState(null,"",`?ticker=${encodeURIComponent(data.meta.symbol)}&tf=${encodeURIComponent(tf)}`);
  }catch(err){
    $("out").innerHTML = `<div class="panel err"><b>Could not run the analysis.</b><br>${esc(err.message)}</div>`;
  }finally{ $("go").disabled = false; }
}

function render(d){
  const m = d.meta, rows = d.indicators;
  const strip = rows.map(r => `<i style="background:${col(r.score)}" title="${esc(r.name)}: ${esc(r.valueText)}" data-n="${r.n}"></i>`).join("");
  const groups = d.groups.map(g => `<div><span>${esc(g.name)}</span><strong>${g.score==null?"-":Math.round(g.score)}</strong></div>`).join("");
  $("out").innerHTML = `
  <section class="panel">
    <div class="hero">
      <div><div class="big">${Math.round(m.overall)}<small>/100</small></div><div class="rate" style="color:${col(m.overall)}">${esc(m.rating)}</div></div>
      <div>
        <div class="name">${esc(m.name)}<span class="sym">${esc(m.symbol)}</span></div>
        <div class="tags"><span>${esc(m.sector)}</span><span>${esc(m.industry)}</span><span>${esc(m.market)} market</span><span>${esc(m.timeframe)}</span>
        <span>${Number(m.price).toLocaleString(undefined,{maximumFractionDigits:2})} ${esc(m.currency)}</span><span>${esc(m.coverage)} indicators scored</span></div>
      </div>
    </div>
    <div class="strip" id="strip">${strip}</div>
    <div class="legend"><span><b style="background:var(--good)"></b>Inside ideal range</span><span><b style="background:var(--mid)"></b>Near range</span><span><b style="background:var(--bad)"></b>Far from range</span><span><b style="background:var(--none)"></b>No data</span></div>
    <div class="groups">${groups}</div>
  </section>
  <section class="panel">
    <div class="tools">
      <div class="filters" id="filters">
        <button type="button" data-f="all" aria-pressed="true">All 27</button>
        <button type="button" data-f="in" aria-pressed="false">Inside range</button>
        <button type="button" data-f="out" aria-pressed="false">Outside range</button>
        <button type="button" data-f="na" aria-pressed="false">No data</button>
      </div>
      <a class="xl" href="/api/export?ticker=${encodeURIComponent(m.symbol)}&timeframe=${encodeURIComponent(m.timeframe)}">Download Excel</a>
    </div>
    <div class="tablewrap"><table>
      <thead><tr><th>#</th><th>Indicator</th><th>Value</th><th>Ideal range (${esc(m.sector)}, ${esc(m.market)})</th><th>Status</th><th>Score</th></tr></thead>
      <tbody id="tb"></tbody>
    </table></div>
  </section>`;
  window._rows = rows;
  drawRows();
  $("filters").addEventListener("click", e => {
    const b = e.target.closest("button"); if(!b) return;
    filter = b.dataset.f;
    document.querySelectorAll("#filters button").forEach(x => x.setAttribute("aria-pressed", x===b));
    drawRows();
  });
  $("strip").addEventListener("click", e => {
    const i = e.target.closest("i"); if(!i) return;
    filter = "all"; drawRows();
    const tr = document.getElementById("r"+i.dataset.n);
    if(tr){ tr.scrollIntoView({block:"center",behavior:"smooth"}); tr.classList.add("flash"); setTimeout(()=>tr.classList.remove("flash"),1600); }
  });
}

function drawRows(){
  const keep = r => filter==="all" || (filter==="in"&&r.status==="Within") || (filter==="out"&&(r.status==="Below"||r.status==="Above")) || (filter==="na"&&r.status==="No data");
  let html = "", last = null;
  for (const r of window._rows.filter(keep)){
    if (r.group !== last){ html += `<tr class="grp"><td colspan="6">${esc(r.group)}</td></tr>`; last = r.group; }
    const cls = "c-" + r.status.split(" ")[0];
    html += `<tr class="row" id="r${r.n}" data-n="${r.n}"><td>${r.n}</td><td>${esc(r.name)}</td><td class="num">${esc(r.valueText)}</td><td>${esc(r.rangeText)}</td>
      <td><span class="chip ${cls}">${esc(r.status)}</span></td>
      <td class="num"><span class="bar"><u style="width:${r.score==null?0:r.score}%;background:${col(r.score)}"></u></span>${r.score==null?"-":Math.round(r.score)}</td></tr>`;
  }
  $("tb").innerHTML = html || `<tr><td colspan="6" class="empty">No indicators match this filter.</td></tr>`;
}
document.addEventListener("click", e => {          // click a row to see its formula
  const tr = e.target.closest("tr.row"); if(!tr) return;
  const nxt = tr.nextElementSibling;
  if (nxt && nxt.classList.contains("detail")) { nxt.remove(); return; }
  const r = window._rows.find(x => x.n == tr.dataset.n);
  tr.insertAdjacentHTML("afterend", `<tr class="detail"><td colspan="6">Formula: ${esc(r.formula)}</td></tr>`);
});

/* ---------- boot ---------- */
drawTF(); drawRecent();
const p = new URLSearchParams(location.search);
if (p.get("tf") && TFS.includes(p.get("tf"))) { tf = p.get("tf"); drawTF(); }
if (p.get("ticker")) { $("q").value = p.get("ticker"); analyze(p.get("ticker")); }
</script>
</body>
</html>


In [ ]:
# 5. Launch the website and print its public link
import importlib, subprocess, threading, re, sys, os
os.chdir("/content/stockscore")
sys.path.insert(0, "/content/stockscore")
from werkzeug.serving import make_server

import engine, app as webapp
importlib.reload(engine); importlib.reload(webapp)       # picks up any edits you made
engine.load_ranges()                                      # creates ideal_ranges.csv on first run

# stop previous run if you re-run this cell
for name in ("_server", "_tunnel"):
    obj = globals().get(name)
    if obj is not None:
        try: obj.shutdown() if name == "_server" else obj.terminate()
        except Exception: pass

_server = make_server("0.0.0.0", 5000, webapp.app, threaded=True)
threading.Thread(target=_server.serve_forever, daemon=True).start()

_tunnel = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://localhost:5000"],
                           stderr=subprocess.PIPE, text=True)
url = None
for line in _tunnel.stderr:
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0); break
threading.Thread(target=lambda: [None for _ in _tunnel.stderr], daemon=True).start()   # keep pipe drained

print("\nYour app is live. Open this link (works on any device while this cell is running):\n")
print("   ", url, "\n")
try:
    from google.colab import output
    output.serve_kernel_port_as_iframe(5000, height=900)   # also shows the site inside the notebook
except Exception:
    pass
